In [167]:
import random
import copy

In [168]:
class TreeNode:
    def __init__(self, state, move=None, player=None):
        self.state = state
        self.move = move
        self.player = player
        self.children = []
        self.value = None

    def add_child(self, child):
        self.children.append(child)

In [169]:
def print_tree(node, prefix="", is_last=True, depth=0, max_depth=3):
    if depth > max_depth:
        return

    connector = "└── " if is_last else "├── "

    if node.move is None:
        label = f"[ROOT | {node.player} | score={round(node.value, 2) if node.value is not None else '?'}]"
    elif node.move == 'Draw':
        label = f"[CHANCE: Draw | score={round(node.value, 2) if node.value is not None else '?'}]"
    else:
        label = f"[{node.player} plays: {node.move} | score={round(node.value, 2) if node.value is not None else '?'}]"

    print(prefix + (connector if prefix else "") + label)

    extension = "    " if is_last else "│   "
    child_prefix = prefix + extension

    for i, child in enumerate(node.children):
        print_tree(child, child_prefix, is_last=(i == len(node.children) - 1), depth=depth + 1, max_depth=max_depth)

In [170]:
class Card:
    def __init__(self, color, value):
      self.color = color
      self.value = value
    def __repr__(self):
      return f"{self.color} {self.value}"
    def __eq__(self, other):
      if isinstance(other, str):
         return False
      return self.color==other.color and self.value==other.value


def deck_generator():
  colors = ['Red', 'Blue', 'Green', 'Yellow']
  deck = []

  for color in colors:
    for number in range(10):       
        deck.append(Card(color,number)) #real uno deck has 2 of each 
        deck.append(Card(color,number))
    deck.append(Card(color,'Skip'))      
    deck.append(Card(color,'Skip'))      

  random.shuffle(deck)
  return deck


deck=deck_generator()
print(f"Deck size: {len(deck)}")

for card in deck[:5]: #seeing a sample
  print(card)


Deck size: 88
Blue 4
Red Skip
Blue 7
Green 8
Green 7


In [171]:
def initial_game_state(deck):
    state={
        'player1_cards':[deck.pop() for _ in range(5)], 
        'player2_cards':[deck.pop() for _ in range(5)], 
        'player3_cards':[deck.pop() for _ in range(5)], 
        'top_card':deck.pop(),
        'deck': deck,
        
        'winner': None,
        'current_player':1,
        'skip':False

    }

    return state

def get_valid_moves(hand, top_card):
    valid_card=[]

    for card in hand:

        if card.color==top_card.color or card.value==top_card.value: #typical uno rule
            valid_card.append(card)

    if not valid_card:
        return ['Draw']
    
    return valid_card

def apply_move(state, move, player_num):
    new_state=copy.deepcopy(state)
    user_display=f'player{player_num}_cards'

    if move=='Draw':
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[user_display].append(drawn)
            print(f"  Player {player_num} draws: {drawn}")
        else:
            print(" The deck is empty! You cannot draw.")
    else:

        new_state[user_display].remove(move)
        new_state['top_card']=move

        if move.value=='Skip':
            new_state['skip']=True

    if len(new_state[user_display])==0:
        new_state['winner']=player_num

    return new_state


    


In [172]:
def count_skips(hand):
    count=0
    for card in hand:
        if card.value=='Skip':
            count+=1
    return count


def defensive_evaluation(state, player_num):

    user_display=f'player{player_num}_cards'
    my_cards=len(state[user_display])

    opp_cards=0

    for p in [1, 2, 3]:

        if p!=player_num:
            opp_cards+=len(state[f'player{p}_cards'])
    opp_cards=opp_cards/2 #taking avg of the opp cards 

    skips=count_skips(state[user_display]
                      )

    #using the given formaualr
    score=50-6*my_cards + 2 * opp_cards + 4 * skips
    return score



def offensive_evaluation(state, player_num):

    user_display= state[f'player{player_num}_cards']
    my_cards=len(user_display)

    opp_cards=0

    for p in [1,2,3]:

        if p!=player_num:
            opp_cards+=len(state[f'player{p}_cards'])
            
    opp_cards= opp_cards/2


    skips=count_skips(user_display)
    score=50-6*my_cards + 2 * opp_cards + 4 * skips
    
    return score

In [173]:
def next_player(current):
    if current==3:
        return 1
    else:
        return current+1 #helps rotate bw the 3 players

In [174]:
def minimax(state, depth, current_player, my_player, move_label=None):
    node = TreeNode(state, move=move_label, player=f"P{current_player} ({'MAX' if current_player == my_player else 'MIN'})")

    if depth==0 or state['winner']is not None: #if depth limit reached or winner found then stop
        node.value = defensive_evaluation(state, my_player)
        return defensive_evaluation(state, my_player), None, node
    
    moves=get_valid_moves(state[f'player{current_player}_cards'], state['top_card'])
    #getting all valid moves


    if current_player==my_player:
        best_score=-9999
        best_move=None   

        for move in moves:
            #simulate
            new_state=apply_move(state, move, current_player)

            #move to next player
            next_p=next_player(current_player)
            
            #evaluate next moves
            score, _, child =minimax(new_state, depth-1, next_p, my_player, str(move))
            child.value = score
            node.add_child(child)

            #choosing the highest score move
            if score>best_score:
                best_score=score
                best_move=move         

        node.value = best_score
        return best_score, best_move, node

    else: #opponent 

        best_score=9999

        for move in moves:
            new_state=apply_move(state, move, current_player)
            next_p=next_player(current_player)

            score, _, child =minimax(new_state, depth-1, next_p, my_player, str(move))
            child.value = score
            node.add_child(child)

            #minimize the score 
            if score<best_score:
                best_score=score

        node.value = best_score
        return best_score, None, node
    


def p1_minimax(state):
    score, move, tree=minimax(state, 3, current_player=1, my_player=1)
    print("Player 1 chooses:", move, " and score:", score)
    return move, tree

In [175]:
def expectimax(state, depth, current_player, my_player, move_label=None):
    node = TreeNode(state, move=move_label, player=f"P{current_player} ({'MAX' if current_player == my_player else 'OPP'})")

    if depth==0 or state['winner'] is not None:
        node.value = offensive_evaluation(state, my_player)
        return offensive_evaluation(state, my_player), None, node

    moves=get_valid_moves(state[f'player{current_player}_cards'],state['top_card'])
    next_p=next_player(current_player)

    if current_player==my_player: #my turn 
        best_score=-9999
        best_move=None

        for move in moves:

            if move=='Draw':
                # chance simulating all possibilites 
                chance_node = TreeNode(state, move='Draw', player='CHANCE')
                deck=state['deck']

                if not deck:
                    score=offensive_evaluation(state, my_player)

                else:

                    total=0
                    sample = deck[:5] if len(deck) > 5 else deck

                    for card in sample:

                        new_state = copy.deepcopy(state)
                        new_state[f'player{my_player}_cards'].append(card)
                        new_state['deck'].remove(card)

                        s, _, child = expectimax(new_state, depth-1, next_p, my_player, str(card))
                        child.value = s
                        chance_node.add_child(child)
                        total += s

                    score=total/len(sample) #each car has equal probability so

                chance_node.value = round(score, 2)
                node.add_child(chance_node)

            else:
                new_state=apply_move(state, move, current_player)
                score, _, child =expectimax(new_state, depth-1, next_p, my_player, str(move))
                child.value = score
                node.add_child(child)

            if score>best_score:
                best_score=score
                best_move=move

        node.value = best_score
        return best_score, best_move, node


    else: #opponent turn random
        if not moves:
            node.value = offensive_evaluation(state, my_player)
            return offensive_evaluation(state, my_player), None, node

        move = random.choice(moves)
        new_state = apply_move(state, move, current_player)

        score, _, child = expectimax(new_state, depth - 1, next_player(current_player), my_player, str(move))
        child.value = score
        node.add_child(child)
        node.value = score
        return score, None, node
    

def p2_expectimax_decision(state):

    score, move, tree=expectimax(state, 3, current_player=2, my_player=2)

    print("Player 2 chooses:", move, "and score:", round(score, 2))
    return move, tree

In [ ]:
def next_player(current, skip=False):
    order = [1, 2, 3]
    idx = order.index(current)
    return order[(idx + 2) % 3] if skip else order[(idx + 1) % 3]


def play_game(mode='simulation'):
    deck = deck_generator()
    state = initial_game_state(deck)
    turn = 0

    print(f"\nUNO Game Started | Mode: {mode.upper()}")
    print(f"Top card: {state['top_card']}\n")

    while state['winner'] is None and turn < 100:
        current = state['current_player']

        if state['skip']:
            print(f"Player {current} is skipped!")
            state['skip'] = False
            state['current_player'] = next_player(current)
            current = state['current_player']

        print(f"\nTurn {turn + 1} — Player {current}'s turn")
        print(f"Top card : {state['top_card']}")
        print(f"P1 ({len(state['player1_cards'])} cards): {state['player1_cards']}")
        print(f"P2 ({len(state['player2_cards'])} cards): {state['player2_cards']}")
        print(f"P3 ({len(state['player3_cards'])} cards): {state['player3_cards']}")
        print(f"Deck: {len(state['deck'])} remaining")

        if current == 1:
            move, tree = p1_minimax(state)
            print_tree(tree)
            state = apply_move(state, move, 1)

        elif current == 2:
            move, tree = p2_expectimax_decision(state)
            print_tree(tree)
            state = apply_move(state, move, 2)

        elif current == 3:
            valid_moves = get_valid_moves(state['player3_cards'], state['top_card'])

        if mode == 'manual':
            print(f"\nYour hand: {state['player3_cards']}")
            valid_moves = get_valid_moves(state['player3_cards'], state['top_card'])
            display_moves = valid_moves if valid_moves == ['Draw'] else valid_moves + ['Draw']
            
            for i, m in enumerate(display_moves):
                print(f"  {i+1}: {m}")
            
            choice = None
            while choice is None:
                raw = input("Your move (enter number): ").strip()
                if raw.isdigit():
                    idx = int(raw) - 1
                    if 0 <= idx < len(display_moves):
                        choice = idx
                    else:
                        print(f"Enter a number between 1 and {len(display_moves)}")
                else:
                    print("Please type a number and press Enter.")
            
            move = display_moves[choice]
            state = apply_move(state, move, 3)

        state['current_player'] = next_player(current, state['skip'])
        state['skip'] = False
        turn += 1

    print("\n" + "="*40)
    if state['winner']:
        print(f"Player {state['winner']} wins in {turn} turns!")
    else:
        print(f"Game drawn after {turn} turns.")
    print("="*40)

In [178]:
#use "manual" to play by yourself

play_game(mode='simulation')


UNO Game Started | Mode: SIMULATION
Top card: Green 0


Turn 1 — Player 1's turn
Top card : Green 0
P1 (5 cards): [Green 1, Yellow 2, Blue 0, Red 1, Yellow 8]
P2 (5 cards): [Blue 2, Yellow 7, Red 9, Green 2, Yellow 5]
P3 (5 cards): [Blue 2, Green 9, Green 2, Red Skip, Green Skip]
Deck: 72 remaining
Player 1 chooses: Green 1  and score: 34.0

Turn 2 — Player 2's turn
Top card : Green 1
P1 (4 cards): [Yellow 2, Blue 0, Red 1, Yellow 8]
P2 (5 cards): [Blue 2, Yellow 7, Red 9, Green 2, Yellow 5]
P3 (5 cards): [Blue 2, Green 9, Green 2, Red Skip, Green Skip]
Deck: 72 remaining
Player 2 chooses: Green 2 and score: 33.0

Turn 3 — Player 3's turn
Top card : Green 2
P1 (4 cards): [Yellow 2, Blue 0, Red 1, Yellow 8]
P2 (4 cards): [Blue 2, Yellow 7, Red 9, Yellow 5]
P3 (5 cards): [Blue 2, Green 9, Green 2, Red Skip, Green Skip]
Deck: 72 remaining

Turn 4 — Player 1's turn
Top card : Green 2
P1 (4 cards): [Yellow 2, Blue 0, Red 1, Yellow 8]
P2 (4 cards): [Blue 2, Yellow 7, Red 9, Yellow 5]
P3 (5 

explanation:

1. Strategy Employed

Minimax (Player 1 – Defensive Strategy):
The Minimax algorithm was implemented with a defensive approach. It assumes that opponents will always make the worst possible move for the AI. Therefore, it focuses on minimizing risk, reducing the number of cards in hand carefully, and using Skip cards to block opponents and control the flow of the game.

Expectimax (Player 2 – Offensive Strategy):
The Expectimax algorithm was implemented with an offensive approach. It assumes that outcomes are probabilistic, especially during the draw action. Instead of worst-case reasoning, it calculates the expected (average) outcome of moves and focuses on aggressively reducing its own cards, even if it involves some risk.

2. Which Algorithm Performed Best

In the simulation, Minimax (Player 1) performed better overall compared to Expectimax (Player 2). It won more games and showed more consistent performance across multiple runs.

3. Reasoning Behind Superior Performance

The superior performance of Minimax can be explained by the following factors:

Worst-case planning: Minimax assumes opponents play optimally to minimize the AI’s score, leading to safer and more reliable decisions.
Better game control: The defensive strategy prioritizes Skip cards, allowing the AI to block opponents and control turn flow effectively.
Low randomness in game: The UNO setup has limited randomness (only in drawing cards). Therefore, Expectimax’s probabilistic advantage is minimal.
Risk in Expectimax: Expectimax relies on average outcomes and assumes random opponent behavior, which can lead to risky decisions that Minimax avoids.
Balanced decision-making: Minimax balances reducing its own cards while preventing opponents from winning, making it more stable.
